# Nationality Detection - Kaggle Training

Attach `jangedoo/utkface-new`, `msambare/fer2013`, and `kaiska/apparel-dataset`. This notebook builds the required class folders in `/kaggle/working`, trains the four models, and writes `/kaggle/working/nationality_models.zip`.

In [ ]:
import os, json, zipfile, random, re, shutil
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED=42
EPOCHS_CLASS=15
EPOCHS_AGE=10
BATCH_SIZE=32
MAX_IMAGES_PER_CLASS=4000  # set to 0 to use every image
INPUT_ROOT=Path('/kaggle/input')
WORK_ROOT=Path('/kaggle/working')
MODEL_DIR=WORK_ROOT/'models'
MODEL_DIR.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

NATIONALITY_CLASSES=['Indian','US_American','African','Other']
NATIONALITY_DISPLAY_CLASSES=['Indian','US/American','African','Other']
EMOTION_CLASSES=['happy','sad','angry','surprised','neutral','fearful','disgusted']
DRESS_CLASSES=['red','blue','green','black','white','yellow','brown','grey','orange','pink','purple']
IMAGE_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
UTK_RE=re.compile(r'^(\d+)_(\d+)_(\d+)_')
UTK_RACE_TO_NATIONALITY={0:'US_American',1:'African',2:'Other',3:'Indian',4:'Other'}

def is_image(path):
    return Path(path).suffix.lower() in IMAGE_EXTS

def reset_dir(path):
    path=Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True,exist_ok=True)
    return path

def link_image(src, dst):
    src=Path(src); dst=Path(dst)
    dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def normalized_token(text):
    return re.sub(r'[^a-z0-9]+','_',str(text).lower()).strip('_')

def image_files(root):
    for p in Path(root).rglob('*'):
        if p.is_file() and is_image(p):
            yield p

def find_exact_class_root(classes):
    expected={c.lower() for c in classes}
    for root, dirs, _ in os.walk(INPUT_ROOT):
        if expected.issubset({d.lower() for d in dirs}):
            return Path(root)
    return None

def write_class_root(name, class_to_paths, classes, require_all=False):
    out=reset_dir(WORK_ROOT/name)
    rng=random.Random(SEED)
    counts={}
    for cls in classes:
        paths=list(dict.fromkeys(class_to_paths.get(cls, [])))
        rng.shuffle(paths)
        if MAX_IMAGES_PER_CLASS and len(paths)>MAX_IMAGES_PER_CLASS:
            paths=paths[:MAX_IMAGES_PER_CLASS]
        cls_dir=out/cls
        cls_dir.mkdir(parents=True,exist_ok=True)
        for i,src in enumerate(paths):
            suffix=Path(src).suffix.lower() or '.jpg'
            link_image(src, cls_dir/f'{i:06d}_{Path(src).stem[:80]}{suffix}')
        counts[cls]=len(paths)
    missing=[k for k,v in counts.items() if v==0]
    if require_all and missing:
        raise FileNotFoundError(f'{name} has no images for required classes: {missing}. Counts: {counts}')
    if sum(counts.values())<2:
        raise FileNotFoundError(f'{name} found too few images. Counts: {counts}')
    print(name, counts)
    return out, counts

def collect_utkface():
    nationality=defaultdict(list)
    age_rows=[]
    for p in image_files(INPUT_ROOT):
        m=UTK_RE.match(p.name)
        if not m:
            continue
        age=int(m.group(1)); race=int(m.group(3))
        label=UTK_RACE_TO_NATIONALITY.get(race)
        if label:
            nationality[label].append(p)
        age_rows.append({'image_path':str(p),'age':age})
    return nationality, age_rows

def prepare_nationality_root():
    manual=find_exact_class_root(NATIONALITY_CLASSES)
    if manual is not None:
        print('Using existing nationality class folders:', manual)
        return manual, None
    nationality, age_rows=collect_utkface()
    if not age_rows:
        raise FileNotFoundError('No UTKFace images found. Attach Kaggle dataset jangedoo/utkface-new or provide nationality folders.')
    root, counts=write_class_root('utkface_nationality_proxy', nationality, NATIONALITY_CLASSES, require_all=True)
    age_csv=WORK_ROOT/'utkface_age_labels.csv'
    pd.DataFrame(age_rows).to_csv(age_csv,index=False)
    print('UTKFace race mapping used as nationality proxy: 3->Indian, 1->African, 0->US_American, 2/4->Other')
    return root, age_csv

EMOTION_ALIASES={
    'happy':'happy','happiness':'happy',
    'sad':'sad','sadness':'sad',
    'angry':'angry','anger':'angry',
    'surprise':'surprised','surprised':'surprised',
    'neutral':'neutral',
    'fear':'fearful','fearful':'fearful',
    'disgust':'disgusted','disgusted':'disgusted',
}
DRESS_ALIASES={
    'red':'red','blue':'blue','green':'green','black':'black','white':'white','yellow':'yellow',
    'brown':'brown','grey':'grey','gray':'grey','orange':'orange','pink':'pink','purple':'purple','violet':'purple',
}

def label_from_text(text, aliases):
    token=normalized_token(text)
    if token in aliases:
        return aliases[token]
    for part in token.split('_'):
        if part in aliases:
            return aliases[part]
    return None

def collect_images_from_alias_dirs(aliases):
    hits=defaultdict(list)
    for root, _, files in os.walk(INPUT_ROOT):
        label=label_from_text(Path(root).name, aliases)
        if not label:
            continue
        for f in files:
            p=Path(root)/f
            if is_image(p):
                hits[label].append(p)
    return hits

def resolve_image_reference(ref, csv_path):
    if pd.isna(ref):
        return None
    if isinstance(ref, (int, np.integer)):
        ref=str(int(ref))
    elif isinstance(ref, (float, np.floating)) and float(ref).is_integer():
        ref=str(int(ref))
    else:
        ref=str(ref).strip()
        if re.fullmatch(r'\d+\.0', ref):
            ref=ref[:-2]
    if not ref:
        return None
    candidates=[]
    raw=Path(ref)
    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.extend([csv_path.parent/raw, csv_path.parent/'images'/raw, INPUT_ROOT/raw])
    if raw.suffix=='':
        more=[]
        for base in candidates:
            for ext in ['.jpg','.jpeg','.png','.bmp','.webp']:
                more.append(Path(str(base)+ext))
                more.append(base.parent/'images'/(base.name+ext))
        candidates.extend(more)
    for c in candidates:
        if c.exists() and c.is_file() and is_image(c):
            return c
    return None

def collect_dress_from_csv():
    hits=defaultdict(list)
    for csv_path in INPUT_ROOT.rglob('*.csv'):
        try:
            df=pd.read_csv(csv_path, low_memory=False)
        except Exception:
            continue
        cols={normalized_token(c):c for c in df.columns}
        color_col=None
        for key in ['colour','color','basecolour','base_color','base_colour']:
            if key in cols:
                color_col=cols[key]; break
        if color_col is None:
            for key,col in cols.items():
                if 'colour' in key or 'color' in key:
                    color_col=col; break
        if color_col is None:
            continue
        image_col=None
        for key in ['image_path','filepath','file_path','filename','file','image','id']:
            if key in cols:
                image_col=cols[key]; break
        if image_col is None:
            continue
        for _,row in df.iterrows():
            label=label_from_text(row[color_col], DRESS_ALIASES)
            if not label:
                continue
            img=resolve_image_reference(row[image_col], csv_path)
            if img is not None:
                hits[label].append(img)
    return hits

def collect_dress_from_filenames():
    hits=defaultdict(list)
    for p in image_files(INPUT_ROOT):
        label=label_from_text(p.parent.name, DRESS_ALIASES) or label_from_text(p.name, DRESS_ALIASES)
        if label:
            hits[label].append(p)
    return hits

def merge_hits(*groups):
    merged=defaultdict(list)
    for group in groups:
        for label, paths in group.items():
            merged[label].extend(paths)
    return merged

def prepare_alias_root(name, classes, aliases, require_all=False, include_csv=False, include_filenames=False):
    exact=find_exact_class_root(classes)
    if exact is not None:
        print(f'Using existing {name} class folders:', exact)
        return exact
    groups=[collect_images_from_alias_dirs(aliases)]
    if include_csv:
        groups.append(collect_dress_from_csv())
    if include_filenames:
        groups.append(collect_dress_from_filenames())
    hits=merge_hits(*groups)
    root, counts=write_class_root(name, hits, classes, require_all=require_all)
    if not require_all:
        missing=[k for k,v in counts.items() if v==0]
        if missing:
            print(f'Warning: {name} has no examples for {missing}; output layer still keeps these classes for app compatibility.')
    return root

def find_age_csv(generated_age_csv):
    for p in INPUT_ROOT.rglob('age_labels.csv'):
        return p
    if generated_age_csv is not None and Path(generated_age_csv).exists():
        return generated_age_csv
    _, age_rows=collect_utkface()
    if age_rows:
        out=WORK_ROOT/'utkface_age_labels.csv'
        pd.DataFrame(age_rows).to_csv(out,index=False)
        return out
    raise FileNotFoundError('No age_labels.csv and no UTKFace filenames found for age training.')

def class_ds(root, classes, size, batch=BATCH_SIZE):
    tr=keras.utils.image_dataset_from_directory(root,validation_split=0.2,subset='training',seed=SEED,image_size=size,batch_size=batch,label_mode='categorical',class_names=classes)
    va=keras.utils.image_dataset_from_directory(root,validation_split=0.2,subset='validation',seed=SEED,image_size=size,batch_size=batch,label_mode='categorical',class_names=classes)
    norm=layers.Rescaling(1./255)
    return tr.map(lambda x,y:(norm(x),y)).prefetch(tf.data.AUTOTUNE), va.map(lambda x,y:(norm(x),y)).prefetch(tf.data.AUTOTUNE)

def cnn(num_classes, input_shape, name):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32,3,activation='relu'),layers.MaxPooling2D(),
        layers.Conv2D(64,3,activation='relu'),layers.MaxPooling2D(),
        layers.Conv2D(128,3,activation='relu'),layers.MaxPooling2D(),
        layers.Flatten(),layers.Dense(256,activation='relu'),layers.Dropout(0.5),
        layers.Dense(num_classes,activation='softmax')], name=name)

def train_classifier(root, classes, size, model_name, out_name, grayscale=False):
    tr,va=class_ds(root,classes,size)
    if grayscale:
        tr=tr.map(lambda x,y:(tf.image.rgb_to_grayscale(x),y))
        va=va.map(lambda x,y:(tf.image.rgb_to_grayscale(x),y))
        input_shape=(size[0],size[1],1)
    else:
        input_shape=(size[0],size[1],3)
    m=cnn(len(classes),input_shape,model_name)
    m.compile(keras.optimizers.Adam(1e-3),loss='categorical_crossentropy',metrics=['accuracy'])
    m.fit(tr,validation_data=va,epochs=EPOCHS_CLASS,callbacks=[keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)])
    loss,acc=m.evaluate(va,verbose=0)
    m.save(MODEL_DIR/out_name)
    return float(acc)

nat_root, generated_age_csv=prepare_nationality_root()
emo_root=prepare_alias_root('fer2013_emotion', EMOTION_CLASSES, EMOTION_ALIASES, require_all=True)
dress_root=prepare_alias_root('apparel_dress_colour', DRESS_CLASSES, DRESS_ALIASES, require_all=False, include_csv=True, include_filenames=True)
print('roots:', nat_root, emo_root, dress_root)

nat_acc=train_classifier(nat_root,NATIONALITY_CLASSES,(128,128),'nationality_detector','nationality_detector.keras')
emo_acc=train_classifier(emo_root,EMOTION_CLASSES,(48,48),'emotion_predictor','emotion_predictor.keras',grayscale=True)
dress_acc=train_classifier(dress_root,DRESS_CLASSES,(128,128),'dress_colour_classifier','dress_colour_classifier.keras')

age_csv=find_age_csv(generated_age_csv)
age_df=pd.read_csv(age_csv)
base=Path(age_csv).parent
paths=[str((base/x).resolve()) if not str(x).startswith('/') else str(x) for x in age_df.image_path.astype(str)]
ages=age_df.age.astype('float32').values
pairs=[(p,a) for p,a in zip(paths,ages) if Path(p).exists()]
if len(pairs)<2:
    raise FileNotFoundError('Age training needs at least two valid images.')
random.Random(SEED).shuffle(pairs)
split=max(1,int(0.2*len(pairs)))
if split>=len(pairs): split=1
val_pairs=pairs[:split]
train_pairs=pairs[split:]

def make_age_ds(pairs, shuffle=False):
    p=[x[0] for x in pairs]
    a=np.array([x[1] for x in pairs],dtype='float32')
    ds=tf.data.Dataset.from_tensor_slices((p,a))
    if shuffle:
        ds=ds.shuffle(len(p),seed=SEED)
    def load_age(path, age):
        img=tf.io.read_file(path)
        img=tf.image.decode_image(img,channels=3,expand_animations=False)
        img=tf.image.resize(img,(128,128))
        return img/255.0, age
    return ds.map(load_age,num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_age_ds(train_pairs, shuffle=True)
val_ds=make_age_ds(val_pairs)
age_model=keras.Sequential([
    layers.Input(shape=(128,128,3)),
    layers.Conv2D(32,3,activation='relu'),layers.MaxPooling2D(),
    layers.Conv2D(64,3,activation='relu'),layers.MaxPooling2D(),
    layers.Conv2D(128,3,activation='relu'),layers.MaxPooling2D(),
    layers.Flatten(),layers.Dense(256,activation='relu'),layers.Dropout(0.3),layers.Dense(1,activation='relu')], name='nationality_age_estimator')
age_model.compile(keras.optimizers.Adam(1e-3),loss='mae',metrics=['mae'])
age_model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS_AGE,callbacks=[keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)])
age_model.save(MODEL_DIR/'age_estimator.keras')

(MODEL_DIR/'nationality_config.json').write_text(json.dumps({
    'seed':SEED,
    'nationality_classes':NATIONALITY_DISPLAY_CLASSES,
    'training_folder_classes':NATIONALITY_CLASSES,
    'emotion_classes':EMOTION_CLASSES,
    'dress_classes':DRESS_CLASSES,
    'nationality_dataset_note':'UTKFace race labels are used as a nationality proxy when no explicit nationality folders are supplied.',
    'validation_accuracy':{'nationality':nat_acc,'emotion':emo_acc,'dress':dress_acc}}, indent=2))

zip_path='/kaggle/working/nationality_models.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in MODEL_DIR.iterdir():
        z.write(p,arcname=p.name)
print('Download:',zip_path)
print(sorted(p.name for p in MODEL_DIR.iterdir()))
